# 04 — Weak Supervision (Snorkel)

Per class, the most TF-IDF-distinctive keywords are auto-derived from that
class's 5% labeled seed (`utils.weak_supervision.derive_class_keywords`) —
a programmatic replacement for hand-written rules, needed because
hand-authoring 16 classes' worth of keyword lists (several overlapping in
subject matter) doesn't scale the way it did for AG News's 4 classes.
Each class's keyword list becomes one labeling function
(`build_keyword_lfs`); Snorkel's `LabelModel` combines all 16 LFs' votes
into probabilistic labels, trained on raw text, no fine-tuning.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd
from snorkel.labeling import LFAnalysis, PandasLFApplier
from snorkel.labeling.model import LabelModel

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_label_quality
from utils.samples import save_full_output
from utils.weak_supervision import derive_class_keywords, build_keyword_lfs

ABSTAIN = -1

In [2]:
labeled_df = pd.read_parquet(config.PROCESSED_DIR / "labeled.parquet")
unlabeled_df = pd.read_parquet(config.PROCESSED_DIR / "unlabeled.parquet")
unlabeled_sample = stratified_sample(unlabeled_df, config.SAMPLE_SIZE, seed=config.SEED, label_col="true_label")

keywords_by_class = derive_class_keywords(labeled_df, config.CLASS_NAMES, top_n=15)
lfs = build_keyword_lfs(keywords_by_class)
print(f"Built {len(lfs)} auto-derived labeling functions from the {len(labeled_df)}-row seed")
for label, keywords in keywords_by_class.items():
    print(f"  {config.CLASS_NAMES[label]}: {keywords[:8]}")

Built 16 auto-derived labeling functions from the 16-row seed
  ARTS & CULTURE: ['lot', 'journey', 'freely', 'life', 'thing', 'sort', 'best', 'pretty']
  BUSINESS: ['coors', 'plastic', 'molson', 'cardboard', 'packaging', 'rings', 'beer', 'hattersley']
  COMEDY: ['meyers', 'trump', 'seth', 'daniels', 'donald', 'president', 'inner', 'serial']
  CRIME: ['bus', 'authority', 'port', 'terminal', 'ullah', 'york', 'police', 'subway']
  EDUCATION: ['joke', 'student', 'speech', 'ramon', 'san', 'issue', 'valley', 'district']
  ENTERTAINMENT: ['meet', 'fockers', 'comedy', 'box', 'uk', 'weekend', 'office', 'role']
  ENVIRONMENT: ['national', 'park', 'historic', 'yosemite', 'site', 'weekend', 'king', 'labor']
  HEALTH: ['vaccine', 'israel', 'booster', 'covid', 'health', 'age', 'data', 'boosters']
  MEDIA: ['fake', 'news', 'smith', 'flynn', 'trump', 'bussey', 'shepard', 'investigation']
  NEWS: ['photos', 'bushfires', 'rage', 'caption', '105', 'hide', 'january', 'australiaa']
  POLITICS: ['florida', 

In [3]:
applier = PandasLFApplier(lfs=lfs)
L_train = applier.apply(df=unlabeled_sample)
L_dev = applier.apply(df=labeled_df)

print(LFAnalysis(L=L_train, lfs=lfs).lf_summary())
print(LFAnalysis(L=L_dev, lfs=lfs).lf_summary(Y=labeled_df["label"].to_numpy()))

  0%|          | 0/303 [00:00<?, ?it/s]

 34%|███▍      | 103/303 [00:00<00:00, 1027.28it/s]

 68%|██████▊   | 206/303 [00:00<00:00, 907.73it/s] 

100%|██████████| 303/303 [00:00<00:00, 926.76it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 891.09it/s]

              j Polarity  Coverage  Overlaps  Conflicts
lf_class_0    0      [0]  0.891089  0.887789   0.887789
lf_class_1    1      [1]  0.844884  0.844884   0.844884
lf_class_2    2      [2]  0.514851  0.514851   0.514851
lf_class_3    3      [3]  0.947195  0.947195   0.947195
lf_class_4    4      [4]  0.716172  0.716172   0.716172
lf_class_5    5      [5]  0.537954  0.537954   0.537954
lf_class_6    6      [6]  0.858086  0.858086   0.858086
lf_class_7    7      [7]  0.864686  0.864686   0.864686
lf_class_8    8      [8]  0.778878  0.778878   0.778878
lf_class_9    9      [9]  0.587459  0.587459   0.587459
lf_class_10  10     [10]  0.699670  0.699670   0.699670
lf_class_11  11     [11]  0.858086  0.858086   0.858086
lf_class_12  12     [12]  0.561056  0.561056   0.561056
lf_class_13  13     [13]  0.891089  0.891089   0.891089
lf_class_14  14     [14]  0.554455  0.554455   0.554455
lf_class_15  15     [15]  0.917492  0.917492   0.917492
              j Polarity  Coverage  Overlaps  Co


C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\scipy\sparse\_construct.py:543: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the output data type will match the input. To avoid this warning, set the `dtype` parameter to `None` to have the output dtype match the input, or set it to the desired output data type.
Note: In Python 3.11, this warning can be generated by a call of scipy.sparse.diags(), but the code indicated in the warning message will refer to an internal call of scipy.sparse.diags_array(). If that happens, check your code for the use of diags().
  A = diags_array(diagonals, offsets=offsets, shape=shape, dtype=dtype)
C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\scipy\sparse\_construct.py:543: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the ou

In [4]:
overall_coverage = (L_train != ABSTAIN).any(axis=1).mean()
# Lower sanity bound than AG News's 4-class version (0.05) — 16
# auto-derived, narrower per-class keyword lists over overlapping
# subject-matter classes are expected to abstain more often.
assert overall_coverage > 0.02, f"LF coverage suspiciously low: {overall_coverage:.2%}"
print(f"Overall LF coverage on unlabeled sample: {overall_coverage:.2%}")

Overall LF coverage on unlabeled sample: 100.00%


In [5]:
label_model = LabelModel(cardinality=config.NUM_CLASSES, verbose=True)
label_model.fit(L_train=L_train, n_epochs=500, lr=0.001, seed=config.SEED)

proba_labels = label_model.predict_proba(L=L_train)
hard_labels = label_model.predict(L=L_train)
confidence = proba_labels.max(axis=1)

INFO:root:Computing O...


INFO:root:Estimating \mu...


  0%|          | 0/500 [00:00<?, ?epoch/s]

INFO:root:[0 epochs]: TRAIN:[loss=97.150]


  1%|          | 5/500 [00:00<00:10, 49.01epoch/s]

...

INFO:root:[480 epochs]: TRAIN:[loss=0.031]


 97%|█████████▋| 484/500 [00:01<00:00, 278.44epoch/s]

INFO:root:[490 epochs]: TRAIN:[loss=0.031]


100%|██████████| 500/500 [00:01<00:00, 254.02epoch/s]


INFO:root:Finished Training


In [6]:
label_quality = evaluate_label_quality(
    true_labels=unlabeled_sample["true_label"].to_numpy(),
    pseudo_labels=hard_labels,
    confidence_scores=confidence)
print("Weak supervision label quality:", label_quality)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_weak_supervision.json", "w") as f:
    json.dump(label_quality, f, indent=2)

Weak supervision label quality: {'Label Accuracy': 0.0627062706270627, 'Label Macro F1': 0.007468553459119497, 'Coverage': np.float64(1.0), 'Mean Confidence': 0.11215065026143704, 'Median Confidence': 0.11454684232418391}


In [7]:
from utils.samples import save_label_samples

save_label_samples(
    unlabeled_sample["text"], hard_labels, unlabeled_sample["true_label"].to_numpy(),
    config.CLASS_NAMES, confidence=confidence, n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_weak_supervision.csv")
print("Saved sample generated labels for weak_supervision.")

save_full_output(
    unlabeled_sample["text"], hard_labels, unlabeled_sample["true_label"].to_numpy(),
    config.CLASS_NAMES, confidence=confidence,
    extra_columns={"summary": unlabeled_sample["summary"].tolist()},
    path=config.RESULTS_DIR / "full_labels_weak_supervision.csv")
print("Saved full-row output for weak_supervision.")

Saved sample generated labels for weak_supervision.
Saved full-row output for weak_supervision.
